In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os
import re

# Handle house price
def convert_price(price_str):
    try:
        price_str = str(price_str).lower().replace(",", ".").strip()

        if 'tỷ' in price_str:
            numeric = ''.join(ch for ch in price_str if ch.isdigit() or ch == '.')
            return float(numeric)

        elif 'triệu' in price_str:
            numeric = ''.join(ch for ch in price_str if ch.isdigit() or ch == '.')
            return float(numeric) / 1000

        return None
    except:
        return None




# Handle outliers
def remove_outliers_iqr(df, columns, multiplier=1.5):
    for col in columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - multiplier * IQR
            upper_bound = Q3 + multiplier * IQR

            # Filter rows within the bounds
            df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

# Folder contains csv files
folder_path = '/content/drive/MyDrive/DA/HCMC_House_Dataset/Raw_Dataset'

# Get files name
districts = ['binh-tan', 'binh-thanh', 'go-vap', 'phu-nhuan']

# Finding files contain value in 'districts'
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv') and any(d in f for d in districts)]

# Header of new data file after cleaning
first_file = True

# Read each file to clean data
for f in csv_files:
  file_path = os.path.join(folder_path, f)
  df = pd.read_csv(file_path)
  df.columns = ["Price", "Address", "Type", "Area", "Width", "Length", "Bedroom", "Restroom", "Floor", "LegalDocument"]

  # Drop rows contain null values
  df.dropna(inplace=True)

  # Price
  df['Price'] = df['Price'].apply(convert_price)

  # Drop luxurious house
  df = df[~df['Type'].str.contains('biệt thự', na=False)]


  # Clean address value
  df['Address'] = df['Address'].str.lower() \
    .str.replace('thành phố hồ chí minh', '', regex=False) \
    .str.replace(', tp hồ chí minh', '', regex=False) \
    .str.replace(', hồ chí minh', '', regex=False) \
    .str.replace('hcm', '', regex=False) \
    .str.replace('quận', '', regex=False) \
    .str.replace('huyện', '', regex=False) \
    .str.replace('phường', '', regex=False) \
    .str.replace('xã', '', regex=False) \
    .str.replace('thị trấn', '', regex=False) \
    .str.title() \
    .str.strip()

  # Keeping rows have Street in it (if its contain both 'Duong' and 'Hem', drop it)
  df = df[
    df['Address'].str.contains(r'\b(đường|hẻm)\b', case=False, na=False) &
    ~df['Address'].str.contains(r'đường.*hẻm|hẻm.*đường', case=False, na=False)
  ]



  # Split address into 3 more attributes: Street, Ward//Commune, District
  df[['Street', 'Ward/Commune', 'District']] = df['Address'].str.split(',', n=2, expand=True)

  # Remove unwanted space
  df['Street'] = df['Street'].str.replace('Đường', '').str.strip()
  df['Ward/Commune'] = df['Ward/Commune'].str.strip()
  df['District'] = df['District'].str.strip()

  # Delete old address column
  df = df.drop(columns=['Address'])

  # Width
  df['Width'] = df['Width'].str.extract(r'([\d.]+)').astype(float).round(2)

  # Length
  df['Length'] = df['Length'].str.extract(r'([\d.]+)').astype(float).round(2)

  # Area
  df['Area'] = (df['Width']*df['Length']).round(2)

  # Bedroom
  df['Bedroom'] = df['Bedroom'].replace('nhiều hơn 10 phòng', '11')
  df['Bedroom'] = df['Bedroom'].fillna(method='ffill')
  df['Bedroom'] = pd.to_numeric(df['Bedroom'].str.extract(r'(\d+)', expand=False), errors='coerce').astype('Int64')


  # Restroom
  df['Restroom'] = df['Restroom'].replace('Nhiều hơn 6 phòng', '7')
  df['Restroom'] = df['Restroom'].fillna(method='ffill')
  df['Restroom'] = pd.to_numeric(df['Restroom'].str.extract(r'(\d+)', expand=False), errors='coerce').astype('Int64')

  # Floor
  df['Floor'] = df['Floor'].fillna(method='ffill')
  df['Floor'] = df['Floor'].astype('Int64')


  # Detect and remove outliers
  df = remove_outliers_iqr(df, ['Floor'])

  # Row after remove outliers
  num_rows = df.shape[0]
  print("Number of rows after:", num_rows)

  clean_file = f.replace('.csv', '')

  # Write to csv file
  df.to_csv(f'/content/drive/MyDrive/DA/HCMC_House_Dataset/Preprocessed_Dataset/Clean_{clean_file}.csv', mode='a', index=False, encoding="utf-8-sig", header=first_file)
  first_file = False

print('Đã lưu ra file!')

In [ ]:
import pandas as pd
import os

# Folder contains csv files
folder_path = '/content/drive/MyDrive/DA/HCMC_House_Dataset/Preprocessed_Dataset'

def remove_outliers_iqr(df, columns):
    """
    Loại bỏ outliers cho các cột được chỉ định trong dataframe theo phương pháp IQR.

    Parameters:
    - df: DataFrame gốc
    - columns: danh sách các tên cột muốn xử lý outlier

    Returns:
    - DataFrame sau khi đã loại bỏ các dòng có outlier
    """
    df_clean = df.copy()

    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = Q3 + 1.5 * IQR

        df_clean = df_clean[df_clean[col] <= upper_bound]

    return df_clean

# Get files name
districts = ['BinhTan', 'BinhThanh', 'GoVap', 'PhuNhuan']

# Finding files contain value in 'districts'
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv') and any(d in f for d in districts)]

# Header of new data file after cleaning
first_file = True

# Read each file to clean data
for f in csv_files:
  file_path = os.path.join(folder_path, f)
  df = pd.read_csv(file_path)
  print(df.isnull().sum())
  null_mask = df.isnull().any(axis=1)
  null_rows = df[null_mask]
  print(null_rows)
  print(df['District'])
#

Price            0
Type             0
Area             0
Width            0
Length           0
Bedroom          0
Restroom         0
Floor            0
LegalDocument    0
Street           0
Ward/Commune     0
District         0
dtype: int64
Empty DataFrame
Columns: [Price, Type, Area, Width, Length, Bedroom, Restroom, Floor, LegalDocument, Street, Ward/Commune, District]
Index: []
0       Bình Thạnh
1       Bình Thạnh
2       Bình Thạnh
3       Bình Thạnh
4       Bình Thạnh
           ...    
2086    Bình Thạnh
2087    Bình Thạnh
2088    Bình Thạnh
2089    Bình Thạnh
2090    Bình Thạnh
Name: District, Length: 2091, dtype: object
Price            0
Type             0
Area             0
Width            0
Length           0
Bedroom          0
Restroom         0
Floor            0
LegalDocument    0
Street           0
Ward/Commune     0
District         0
dtype: int64
Empty DataFrame
Columns: [Price, Type, Area, Width, Length, Bedroom, Restroom, Floor, LegalDocument, Street, Ward/Commune,